Дані й вхід

In [9]:
import os, re, glob, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import QuantileTransformer

os.makedirs("results", exist_ok=True)
dev = torch.device("cuda")

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

N = len(df)
TRAIN_W, VAL_W, TEST_W = (0, 17), (18, 21), (22, 26)
inw = lambda r: ((df.week >= r[0]) & (df.week <= r[1])).values
m_tr, m_va, m_te = inw(TRAIN_W), inw(VAL_W), inw(TEST_W)
assert (m_tr | m_va | m_te).all() and not (m_tr & m_va).any() and not (m_va & m_te).any()
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)

ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts))
rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
y = df.isFraud.values
assert y.sum() == 20663
split_of = np.where(m_tr, 0, np.where(m_va, 1, 2))

X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
df["D1n"] = (df["D1"] - df["TransactionDT"] / 86400).round()   # після X, у X не потрапляє
spw = (y[m_tr] == 0).sum() / (y[m_tr] == 1).sum()

Xn = X.values.astype("float32")
grp = lambda p: [i for i, c in enumerate(X.columns) if re.fullmatch(p, c)]
nanf = np.stack([np.isnan(Xn[:, g]).mean(1) for g in
                 (grp(r"V\d+"), grp(r"D\d+"), grp(r"id_\d+"), grp(r"M\d+"))], 1).astype("float32")
qt = QuantileTransformer(n_quantiles=200, output_distribution="normal",
                         subsample=200_000, random_state=0).fit(Xn[m_tr])
Xt = np.hstack([np.nan_to_num(qt.transform(Xn), nan=0.0).clip(-5, 5).astype("float32"), nanf])
XG = torch.from_numpy(Xt).half().to(dev)       # вхід вузла: лише таблиця, без міток

def metrics(y_true, p):
    k = int(y_true.sum()); o = np.argsort(-p)
    top1 = o[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[o[:k]].sum() / k, "precision@1%": y_true[top1].mean()}
print("готово:", tuple(XG.shape))

готово: (590540, 438)


Граф u2, u3, u5

In [10]:
def build_edges(cols, K, W_days):
    valid = df[cols].notna().all(axis=1).values
    idx = np.flatnonzero(valid); sub = df.loc[valid, cols]
    S, D = [], []
    for pos in sub.groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt = ts[rows]; n = len(rows)
        lo = np.maximum(np.arange(n) - K, np.searchsorted(tt, tt - W_days * 86400, side="left"))
        cnt = np.arange(n) - lo; tot = int(cnt.sum())
        if tot == 0: continue
        dst_l = np.repeat(np.arange(n), cnt)
        start = np.repeat(np.cumsum(cnt) - cnt, cnt)
        src_l = np.arange(tot) - start + np.repeat(lo, cnt)
        S.append(rows[src_l].astype(np.int32)); D.append(rows[dst_l].astype(np.int32))
    return np.concatenate(S), np.concatenate(D)

DL = 7
REL = {"u2": (["card1", "addr1"], 50, 30),
       "u3": (["card1", "addr1", "P_emaildomain"], 50, 30),
       "u5": (["card1", "addr1", "P_emaildomain", "D1n"], 50, 30)}
R_NAMES = list(REL); KMAX = {k: v[1] for k, v in REL.items()}
edges = {k: build_edges(*v) for k, v in REL.items()}
for k, (s, d) in edges.items():
    assert (rank[s] < rank[d]).all() and (s != d).all()
    assert (split_of[s[split_of[d] == 0]] == 0).all()
    print(k, f"ребер {len(s):,}")

NBR, DEG = [], []
for rn in R_NAMES:
    s, d = edges[rn]
    o = np.argsort(d, kind="stable"); ds, ss = d[o], s[o]
    cnt = np.bincount(d, minlength=N); start = np.cumsum(cnt) - cnt
    pos = np.arange(len(ds)) - start[ds]
    assert pos.max() < KMAX[rn]
    tab = np.full((N, KMAX[rn]), -1, dtype=np.int32); tab[ds, pos] = ss
    NBR.append(torch.from_numpy(tab).to(dev))
    DEG.append(torch.from_numpy(cnt.astype(np.int64)).to(dev))

u2 ребер 12,260,904
u3 ребер 5,295,877
u5 ребер 478,008


Модель і тести на витік

In [11]:
ts_t = torch.as_tensor(ts, device=dev)
ST = {"y": torch.as_tensor(y, device=dev).float()}
DLS = DL * 86400

def sample(nodes, r, s, gen=None):
    deg = DEG[r][nodes]
    u = torch.rand(nodes.shape[0], s, device=dev, generator=gen)
    idx = (u * deg.unsqueeze(1)).long().clamp_(max=KMAX[R_NAMES[r]] - 1)
    nb = NBR[r][nodes.unsqueeze(1), idx].long()
    valid = (deg > 0).unsqueeze(1).expand(-1, s) & (nb >= 0)
    return nb.clamp(min=0), valid

def hops(tgt, s1, s2, gen=None):
    B, R = tgt.shape[0], len(R_NAMES)
    n1, v1 = zip(*[sample(tgt, r, s1, gen) for r in range(R)])
    N1, M1 = torch.stack(n1, 1), torch.stack(v1, 1)
    n2, v2 = zip(*[sample(N1.reshape(-1), r, s2, gen) for r in range(R)])
    N2 = torch.stack(n2, 1).view(B, R, s1, R, s2)
    M2 = torch.stack(v2, 1).view(B, R, s1, R, s2) & M1[:, :, :, None, None]
    return N1, M1, N2, M2

def edge_feats(t_dst, nb, valid):
    """ознаки ребра: log(1+днів), мітка відома, відомий фрод, відомий чесний"""
    gap = t_dst - ts_t[nb]
    known = valid & (gap >= DLS)
    yl = ST["y"][nb]
    f = torch.stack([torch.log1p(gap.clamp(min=0).float() / 86400.0),
                     known.float(), (known & (yl > 0.5)).float(), (known & (yl < 0.5)).float()], -1)
    return f * valid.unsqueeze(-1).float()

class RelSAGE2E(nn.Module):
    def __init__(self, F_in, H=128, R=3, drop=0.2):
        super().__init__()
        self.R = R
        self.enc = nn.Sequential(nn.Linear(F_in, H), nn.ReLU(), nn.Dropout(drop))
        self.W1s, self.W2s = nn.Linear(H, H), nn.Linear(H, H)
        self.Wm1 = nn.ModuleList([nn.Linear(H + 4, H) for _ in range(R)])
        self.Wm2 = nn.ModuleList([nn.Linear(H + 4, H) for _ in range(R)])
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(H, 1))

    @staticmethod
    def mmean(e, m, dim):
        m = m.unsqueeze(-1).to(e.dtype)
        return (e * m).sum(dim) / m.sum(dim).clamp(min=1)

    def forward(self, XG, tgt, s1, s2, gen=None):
        B, R = tgt.shape[0], self.R
        N1, M1, N2, M2 = hops(tgt, s1, s2, gen)
        f1 = edge_feats(ts_t[tgt][:, None, None], N1, M1)
        f2 = edge_feats(ts_t[N1][:, :, :, None, None], N2, M2)
        enc = lambda ids: self.enc(XG[ids].float())
        e_t = enc(tgt)
        e1 = enc(N1.reshape(-1)).view(B, R, s1, -1)
        e2 = enc(N2.reshape(-1)).view(B, R, s1, R, s2, -1)
        agg = 0
        for r in range(R):
            m = F.relu(self.Wm1[r](torch.cat([e2[:, :, :, r], f2[:, :, :, r]], -1)))
            agg = agg + self.mmean(m, M2[:, :, :, r], 3)
        h1_u = F.relu(self.W1s(e1) + agg)
        agg = 0
        for r in range(R):
            m = F.relu(self.Wm1[r](torch.cat([e1[:, r], f1[:, r]], -1)))
            agg = agg + self.mmean(m, M1[:, r], 1)
        h1_t = F.relu(self.W1s(e_t) + agg)
        agg = 0
        for r in range(R):
            m = F.relu(self.Wm2[r](torch.cat([h1_u[:, r], f1[:, r]], -1)))
            agg = agg + self.mmean(m, M1[:, r], 1)
        return self.out(F.relu(self.W2s(h1_t) + agg)).squeeze(-1)

# ---- тест 1: сусіди з минулого й зі своєї частини розбиття ----
sp_t = torch.as_tensor(split_of, device=dev)
rg = np.random.default_rng(0)
for sp, cap in [(0, 0), (1, 1)]:
    ids = torch.as_tensor(rg.choice(np.flatnonzero(split_of == sp), 4000, replace=False), device=dev)
    N1, M1, N2, M2 = hops(ids, 10, 5)
    assert (sp_t[N1[M1]] <= cap).all() and (sp_t[N2[M2]] <= cap).all(), "сусіди з майбутньої частини"
    assert (ts_t[N1] <= ts_t[ids][:, None, None])[M1].all(), "сусід пізніший за вузол (крок 1)"
    assert (ts_t[N2] <= ts_t[N1][:, :, :, None, None])[M2].all(), "сусід пізніший за вузол (крок 2)"
print("тест 1 пройдено")

# ---- тест 2: невідомі мітки не впливають на вихід, відомі впливають ----
te_nodes = np.flatnonzero(m_te)
T0 = ts[te_nodes[len(te_nodes) // 2]]
win = te_nodes[(ts[te_nodes] >= T0) & (ts[te_nodes] <= T0 + 86400)][:1500]
assert len(win) > 500
Tcut = T0 + 86400 - DLS                      # мітки вузлів із ts > Tcut не відомі жодній цілі вікна
tgt_w = torch.as_tensor(win, device=dev); y0 = ST["y"].clone()

def logits(mdl, yy):
    ST["y"] = yy
    g = torch.Generator(device=dev); g.manual_seed(7)
    with torch.no_grad(): out = mdl(XG, tgt_w, 10, 5, g)
    ST["y"] = y0
    return out

torch.manual_seed(0)
mdl = RelSAGE2E(XG.shape[1]).to(dev).eval()
unk = y0.clone(); mu = ts_t > Tcut; unk[mu] = 1 - unk[mu]
kn = y0.clone(); mk_ = ts_t <= Tcut; kn[mk_] = 1 - kn[mk_]
a, b, c = logits(mdl, y0), logits(mdl, unk), logits(mdl, kn)
assert torch.allclose(a, b, atol=1e-6), "невідомі мітки впливають на вихід (ВИТІК)"
assert not torch.allclose(a, c, atol=1e-6), "відомі мітки не впливають: модель їх не бачить"
print("тест 2 пройдено | параметрів:", sum(p.numel() for p in mdl.parameters()))

тест 1 пройдено
тест 2 пройдено | параметрів: 191489


Пробний запуск (seed 0)

In [12]:
@torch.no_grad()
def predict(model, idx_np, s1, s2, seed, bs=2048):
    model.eval(); ids = torch.as_tensor(idx_np, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(1234 + 100 * seed)
    return torch.cat([torch.sigmoid(model(XG, ids[i:i + bs], s1, s2, gen)).float().cpu()
                      for i in range(0, len(ids), bs)]).numpy()

def train_edge(seed, s1=10, s2=5, epochs=15, patience=4, bs=1024, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = RelSAGE2E(XG.shape[1]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    posw = torch.tensor(float(spw), device=dev)
    tr_idx = torch.as_tensor(np.flatnonzero(m_tr), device=dev)
    yt = torch.as_tensor(y, device=dev).float()
    va_idx, te_idx = np.flatnonzero(m_va), np.flatnonzero(m_te)
    best, best_ep, bad, state, t_all = -1.0, 0, 0, None, time.time()
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time()
        perm = tr_idx[torch.randperm(len(tr_idx), device=dev)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(model(XG, b, s1, s2), yt[b], pos_weight=posw)
            opt.zero_grad(); loss.backward(); opt.step()
        v = average_precision_score(y[va_idx], predict(model, va_idx, s1, s2, seed))
        print(f"seed {seed} епоха {ep}: val PR-AUC {v:.4f} | {time.time()-t0:.0f} с", flush=True)
        if v > best:
            best, best_ep, bad = v, ep, 0
            state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    r = metrics(y[m_te], predict(model, te_idx, s1, s2, seed))
    r.update(seed=seed, best_epoch=best_ep, val_aucpr=best, secs=time.time() - t_all)
    return r

r0 = train_edge(0)
pd.DataFrame([r0]).to_csv("results/ieee_cis_edge_labels_seed0.csv", index=False)
print({k: round(float(v), 4) for k, v in r0.items() if k in ("pr_auc", "roc_auc", "recall@k", "precision@1%", "val_aucpr")})
print("правило: val GNN з мітками на ребрах > 0,628 →",
      "ПРОЙДЕНО, запускай комірку 5" if r0["val_aucpr"] > 0.628 else "не пройдено, зупиняємось")

seed 0 епоха 1: val PR-AUC 0.5146 | 48 с
seed 0 епоха 2: val PR-AUC 0.5728 | 47 с
seed 0 епоха 3: val PR-AUC 0.5863 | 47 с
seed 0 епоха 4: val PR-AUC 0.5972 | 47 с
seed 0 епоха 5: val PR-AUC 0.5783 | 47 с
seed 0 епоха 6: val PR-AUC 0.5818 | 46 с
seed 0 епоха 7: val PR-AUC 0.5910 | 47 с
seed 0 епоха 8: val PR-AUC 0.5857 | 47 с
{'pr_auc': 0.5796, 'roc_auc': 0.9226, 'recall@k': 0.5565, 'precision@1%': 0.8761, 'val_aucpr': 0.5972}
правило: val GNN з мітками на ребрах > 0,628 → не пройдено, зупиняємось
